In [6]:
import pandas as pd
import glob
import os
from datetime import timedelta
from pandas.tseries.offsets import BDay
import numpy as np

In [7]:
df_OTIF         = pd.read_excel("MasterBatchTracker.xlsx", index_col=0)
df_QC_LT        = pd.read_excel("SKU_LAB.xlsx")
df_Release_SAP  = pd.read_excel("MB51_Release_dates.xlsx")
df_E2E          = pd.read_excel("E2E_analysis.xlsx")
df_E2E          = df_E2E[(df_E2E["TPT"]=="TPT7 - FP release time") & (df_E2E["Plant"]=="MX01") ]
df_E2E_Master   = df_E2E.merge(df_OTIF, left_on= "Batch", right_on="LOTE PT")

df_OTIF_release=df_OTIF.merge(df_Release_SAP, on="LOTE PT", how="outer")
df_OTIF_release.keys()
df_OTIF_release["Last_Expedient"] = df_OTIF_release[['FIN FABRI REAL', 'PT_ENTREGA EXPEDIENTE REAL', 'ENTREGA ÓRDEN ANALÍTICA REAL']].max(axis=1)
df_OTIF_release['Dictamen_Release_Plan'] = df_OTIF_release['Last_Expedient'] + pd.offsets.BDay(3)

In [8]:
import pandas as pd

def identify_late_deliveries(df):
    # Create a list to hold late delivery records
    late_deliveries = []

    # Define the conditions for late deliveries
    conditions = [
        ('FIN FABRI PLAN', 'FIN FABRI REAL', 'Fabri'), # Listo
        ('PR_ENTREGA MUESTRA PLAN', 'PR_ENTREGA MUESTRA REAL', 'Muestra'),
        ('PR_ENTREGA EXPEDIENTE GR PLAN', 'PR_ENTREGA EXPEDIENTE GR REAL', 'Revisores Fabri'),
        ('ENTREGA ÓRDEN ANALÍTICA PLAN', 'ENTREGA ÓRDEN ANALÍTICA REAL', 'QC'),
        ('ACO_FIN ACONDI PLAN', 'ACO_FIN ACONDI REAL', 'Acondi'),
        ('PT_ENTREGA EXPEDIENTE PLAN', 'PT_ENTREGA EXPEDIENTE REAL', 'Revisores Acondi'),
        ('FECHA LIBERACION V0','Deviation Close Date', ' Desviación'),
    
    ]
    # Entrega de muestra Plan vs Fecha de liberación PLAN , si Entrega de muestra tarde , entonces no es culpa de planeación 
    # Eutirox <22
    # Sols <21
    # Liq <18, iliadin 11 dias

    # Fin de acondi Plan a Fecha de Liberacion Plan>= 16, mal planeación 

    for plan_col, real_col, bad_actor in conditions:
        # Convert columns to datetime
        df[plan_col] = pd.to_datetime(df[plan_col], errors='coerce')
        df[real_col] = pd.to_datetime(df[real_col], errors='coerce')

        # Filter rows where both planned and real dates are available
        mask = df[plan_col].notna() & df[real_col].notna()
        filtered_df = df[mask]

        for index, row in filtered_df.iterrows():
            if row[real_col] > row[plan_col]:  # Check if delivery is late
                # Create a new record for late delivery
                late_delivery = row.to_dict()
                late_delivery['Bad Actor'] = bad_actor
                late_deliveries.append(late_delivery)

    # Create a DataFrame for late deliveries
    late_deliveries_df = pd.DataFrame(late_deliveries)

    return late_deliveries_df


In [9]:
df_badactors= identify_late_deliveries(df_OTIF_release.copy())

df_badactors.to_excel("BadActors_PLAN.xlsx")



df_E2E_badactors = df_E2E_Master.merge(df_badactors[[ 'LOTE PT', 'LOTE DE GRANEL', 'LOTE DE GRANEL 2',
       'Bad Actor']], left_on="Batch", right_on="LOTE PT", how="left")
df_E2E_badactors["E2E_P2R"] = (df_E2E_badactors["Target"]>=df_E2E_badactors["Lead_time"]).astype(int)
df_E2E_badactors.to_excel("BadActors_E2E_consolidated.xlsx")



# % Cumplimiento de LT en Planes

In [37]:
def check_lead_time_respect(df):
    # Define the lead times in days and corresponding actor names
    lead_times = {
        ('FIN FABRI PLAN', 'PR_ENTREGA EXPEDIENTE GR PLAN'): (7, 'Rev.Fabri'),
        ('PR_ENTREGA MUESTRA PLAN', 'ENTREGA ÓRDEN ANALÍTICA PLAN'): (20, 'QC'),
        ('ACO_FIN ACONDI PLAN', 'PT_ENTREGA EXPEDIENTE PLAN'): (4, 'Rev.Acondi'),
        ('ENTREGA ÓRDEN ANALÍTICA PLAN', 'FECHA LIBERACION V0'): (2, 'Dictamen QC'),
        ('PR_ENTREGA EXPEDIENTE GR PLAN', 'FECHA LIBERACION V0'): (16, 'Dictamen FABRI'),
        ('PT_ENTREGA EXPEDIENTE PLAN', 'FECHA LIBERACION V0'): (5, 'Dictamen PT'), 
        ('ACO_FIN ACONDI PLAN', 'FECHA LIBERACION V0'): (16, 'Planning Acondi'), 
        ('PR_ENTREGA MUESTRA PLAN', 'FECHA LIBERACION V0'): (16, 'Planning Fabri')
    }

    # Convert relevant columns to datetime, coercing errors to NaT
    for plan_col, comparison_col in lead_times.keys():
        df[plan_col] = pd.to_datetime(df[plan_col], errors='coerce')
        df[comparison_col] = pd.to_datetime(df[comparison_col], errors='coerce')

    # Iterate through the lead times to calculate differences and check respect
    for (plan_col, comparison_col), (lead_time, actor_name) in lead_times.items():
        # Create new column names
        diff_col_name = f'Diff {actor_name}'
        respect_col_name = f'LT {actor_name}'

        # Check if both columns have non-null values
        mask = df[plan_col].notna() & df[comparison_col].notna()

        # Calculate the difference in days only for non-null values
        df.loc[mask, diff_col_name] = (df.loc[mask, comparison_col] - df.loc[mask, plan_col]).dt.days

        # Check if lead time is respected only for non-null values
        df.loc[mask, respect_col_name] = (df.loc[mask, diff_col_name] >= lead_time).astype(int)

    return df



In [38]:
df_MasterPlan=check_lead_time_respect(df_OTIF_release.copy())
df_MasterPlanLT=df_MasterPlan.merge(df_QC_LT,on="SKU")
df_MasterPlanLT.keys()
df_MasterPlanLT["LT QC Exact"]= (df_MasterPlanLT["Diff QC"]>= df_MasterPlanLT["LT_Teorico"]).astype(int)
# Assuming df_MasterPlanLT is your DataFrame
df_MasterPlanLT["LT Planning Fabri Exact"] = 0  # Initialize the new column with 0s

# Create a mask for non-null values in the relevant columns
mask = df_MasterPlanLT["Diff Planning Fabri"].notna() & df_MasterPlanLT["LT_FabriPlan"].notna()

# Assign 1 where the condition is met
df_MasterPlanLT.loc[mask, "LT Planning Fabri Exact"] = (df_MasterPlanLT.loc[mask, "Diff Planning Fabri"] >= df_MasterPlanLT.loc[mask, "LT_FabriPlan"]).astype(int)
df_MasterPlanLT.to_excel("LeadTime_analysis.xlsx")

In [39]:
df_badactors_LT=identify_late_deliveries(df_MasterPlanLT.copy())
df_badactors_LT.keys()
df_badactors_LT['Planning Adherence'] = np.where(
    (df_badactors_LT['Bad Actor'] == 'QC') & (df_badactors_LT['LT QC Exact'] == 0),
    'Bad Planning',
    'Adherent'
)
df_badactors_LT.to_excel("Bad_actors_adherance.xlsx")

# BadActor by column

In [1]:
import pandas as pd

def identify_late_deliveries_columns(df):
    # Define the conditions for late deliveries
    conditions = [
        ('FIN FABRI PLAN', 'FIN FABRI REAL', 'Fabri'), 
        ('PR_ENTREGA MUESTRA PLAN', 'PR_ENTREGA MUESTRA REAL', 'Muestra'),
        ('PR_ENTREGA EXPEDIENTE GR PLAN', 'PR_ENTREGA EXPEDIENTE GR REAL', 'Revisores Fabri'),
        ('ENTREGA ÓRDEN ANALÍTICA PLAN', 'ENTREGA ÓRDEN ANALÍTICA REAL', 'QC'),
        ('ACO_FIN ACONDI PLAN', 'ACO_FIN ACONDI REAL', 'Acondi'),
        ('PT_ENTREGA EXPEDIENTE PLAN', 'PT_ENTREGA EXPEDIENTE REAL', 'Revisores Acondi')
    ]

    # Initialize new columns for each bad actor
    for _, _, bad_actor in conditions:
        df[bad_actor] = 0

    # Process each condition
    for plan_col, real_col, bad_actor in conditions:
        # Convert columns to datetime
        df[plan_col] = pd.to_datetime(df[plan_col], errors='coerce')
        df[real_col] = pd.to_datetime(df[real_col], errors='coerce')

        # Filter rows where both planned and real dates are available
        mask = df[plan_col].notna() & df[real_col].notna()
        filtered_df = df[mask]

        for index, row in filtered_df.iterrows():
            if row[real_col] > row[plan_col]:  # Check if delivery is late
                df.at[index, bad_actor] = 1  # Mark bad actor column as 1 for late delivery

    return df


In [16]:
df_badactors_column= identify_late_deliveries_columns(df_OTIF).copy()
df_badactors_column.to_excel("BadActors_columns.xlsx")